# Visual MOSI — Bảo Hoàng — bản báo cáo có code đầy đủ

**Cách đọc:** mỗi task có mục đích, câu nói báo cáo, hàm và đoạn code thật để chỉ trực tiếp. Hai hàm từng nằm ngoài notebook nay nằm đầy đủ tại **Task 2.12A–B** và được các task bên dưới sử dụng.

**1. Mô hình và đặc trưng:** Em phụ trách MAG-BERT và trích xuất visual. MAG dùng audio/visual điều tiết biểu diễn ngôn ngữ BERT để dự đoán sentiment. Nhóm dùng BERT, openSMILE eGeMAPSv02 LLD và OpenFace; visual 22 chiều là cấu hình nhóm, khác Facet trong paper.

**2. Quy trình:** kiểm tra manifest → chuẩn bị cấu hình 22 chiều → chạy thử 10 clip → kiểm tra CSV/ảnh → trích toàn bộ clip và tổng hợp chất lượng.

**3. Kết quả đã báo cáo:** 2.199 clip, 2.139 đạt tiêu chí tự động và 60 cần xem thêm; chưa tự loại. Số liệu cần được xác nhận lại bằng báo cáo thật ở Task 2.21. Không có kết quả giả được điền sẵn vào output.

**4. Tiếp theo:** xử lý thiếu, chuẩn hóa từ train, căn chỉnh theo từ/token cho MAG-BERT. Chưa huấn luyện mô hình.

| Mở trên Drive (tên cũ giữ nguyên) | Task mới trong notebook | Hàm/code cần chỉ |
|---|---|---|
| File 01 — manifest | 2.3 | `read_manifest()` |
| File 02 — pilot | 2.14–2.15 | `extract_one_clip()`, `extract_pilot_clips()` |
| Thư mục 03 — ảnh/video minh họa | 2.16–2.17 | `show_sample_frames()`, `create_visual_review_form()` |
| File 04 — cấu hình 22 chiều | 2.11; 2.12A; 2.18 | `FEATURE_NAMES`; `numeric[names]`; `dtype=np.float32`; `observed_mask` |
| File 06 — kết quả toàn bộ | 2.21–2.22 | `read_saved_results()`, `extract_full_dataset()` |
| Code gọi OpenFace | 2.12B | `command`, `-aus/-pose/-gaze`, `subprocess.run()` |

Thư mục Drive: `CMU-MOSI/preprocessing/visual/Task_2_BaoCao_Visual`. Task 2.2 liệt kê tên file thực tế. Task mới 2.11 tương ứng cấu hình cũ 2.18; task mới 2.18 tương ứng đọc CSV cũ 2.19. Không phải mọi file báo cáo là do cell mới vừa tạo.

**Mặc định chỉ đọc dữ liệu đã có:** `SETUP_OPENFACE=False`, `RUN_PILOT=False`, `RUN_EXTRACTION=False`, `VERIFY_SAVED_FEATURES=False`. Chưa chạy lại trên Colab ở thời điểm tạo bản này. Dữ liệu cũ trên Drive không bị thay đổi.


## Thư viện và cấu hình chạy
Nạp thư viện/đường dẫn. Giữ bốn cờ False để đọc kết quả cũ. Danh sách 22 chiều được ghi lại ngay Task 2.11 để dễ chỉ khi báo cáo.

In [ ]:
# Thư viện và đường dẫn dùng chung
import hashlib
import importlib.util
import json
import os
import platform
import shutil
import subprocess
import sys
import tarfile
import tempfile
import time
import traceback
import types
import zipfile
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath

import numpy as np
import pandas as pd

try:
    import cv2
except ImportError:
    cv2 = None
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(items, **kwargs):
        class Progress:
            def __iter__(self):
                return iter(items)
            def set_postfix(self, **values):
                print(values, flush=True)
        return Progress()
try:
    from IPython.display import display, Video
except ImportError:
    display = print
    Video = None
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
try:
    from google.colab import drive
except ImportError:
    drive = None

SOURCE_COMMIT = '3d4b5cf8d96138be42bed229447f36cbb09a5a29'
SOURCE_URL = 'https://github.com/TadasBaltrusaitis/OpenFace.git'
PROCESSOR_SHA256 = '300b86b4a706c0a246ef614e0f67d7074c99aff90e97f241e52354ff554919ad'
SAVED_SIGNATURE = '3c908e7632a119a9bf6a88d0b4c96d7f21465ed35a57619483811ff9addc17a0'
COMPLETED_RUN = '20261102T140801_205739Z'
AU_NAMES = [f'AU{i:02d}_r' for i in [1,2,4,5,6,7,9,10,12,14,15,17,20,23,25,26,45]]
POSE_NAMES = ['pose_Rx', 'pose_Ry', 'pose_Rz']
GAZE_NAMES = ['gaze_angle_x', 'gaze_angle_y']
FEATURE_NAMES = AU_NAMES + POSE_NAMES + GAZE_NAMES
MODEL_URLS = {
    'cen_patches_0.25_of.dat': 'https://www.dropbox.com/s/7na5qsjzz8yfoer/cen_patches_0.25_of.dat?dl=1',
    'cen_patches_0.35_of.dat': 'https://www.dropbox.com/s/k7bj804cyiu474t/cen_patches_0.35_of.dat?dl=1',
    'cen_patches_0.50_of.dat': 'https://www.dropbox.com/s/ixt4vkbmxgab1iu/cen_patches_0.50_of.dat?dl=1',
    'cen_patches_1.00_of.dat': 'https://www.dropbox.com/s/2t5t1sdpshzfhpj/cen_patches_1.00_of.dat?dl=1',
}

@dataclass(frozen=True)
class VisualPaths:
    data_root: Path = Path('/content/drive/MyDrive/CMU-MOSI')
    openface_root: Path = Path('/content/OpenFace')
    local_root: Path = Path('/content/mosi_visual')

    @property
    def visual(self):
        return self.data_root / 'preprocessing/visual'
    @property
    def shared(self):
        return self.data_root / 'preprocessing/shared'
    @property
    def manifest(self):
        return self.shared / 'manifest.csv'
    @property
    def raw_zip(self):
        return self.data_root / 'Raw.zip'
    @property
    def module(self):
        return self.data_root / 'src/preprocessing/vision_processor.py'
    @property
    def config(self):
        return self.visual / 'config/visual_feature_config.json'
    @property
    def binary(self):
        return self.openface_root / 'build/bin/FeatureExtraction'
    @property
    def model(self):
        return self.binary.parent / 'model/main_ceclm_general.txt'
    @property
    def backup(self):
        return self.visual / 'tools/OpenFace_colab.tar.gz'
    @property
    def full_root(self):
        return self.visual / 'full_visual_22' / SAVED_SIGNATURE[:12]


def file_sha256(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.part')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')
    temporary.replace(path)


def safe_child(root, relative):
    root = Path(root).resolve()
    candidate = (root / str(relative)).resolve()
    if not candidate.is_relative_to(root):
        raise ValueError(f'Đường dẫn nằm ngoài thư mục: {relative}')
    return candidate


def require_cv2():
    if cv2 is None:
        raise RuntimeError('Cần opencv-python cho chức năng đọc video; chế độ reuse không cần.')



SETUP_OPENFACE = False
RUN_PILOT = False
RUN_EXTRACTION = False
VERIFY_SAVED_FEATURES = False
PILOT_RUN = '20261102T070035_074030Z'

## Task 2.1 — Kết nối Drive
**Mục đích:** truy cập dữ liệu CMU-MOSI đã lưu trong tài khoản Drive.
**Hàm:** `connect_drive()`; lệnh quan trọng `drive.mount('/content/drive')`.
Kết nối Drive không phải cài OpenFace hoặc huấn luyện mô hình.

In [ ]:
def connect_drive(mount=False):
    if mount:
        if drive is None:
            raise RuntimeError('Mount Drive chỉ dùng trong Google Colab.')
        drive.mount('/content/drive')
    print('Python:', platform.python_version(), '| NumPy:', np.__version__, '| pandas:', pd.__version__)


# Thực hiện
connect_drive(mount=True)

## Task 2.2 — Xem đường dẫn dữ liệu và file báo cáo
**Mục đích:** biết chính xác mở file nào khi báo cáo.
**Hàm:** `show_data_paths()` đọc các đường dẫn; `show_report_files()` liệt kê tên file/thư mục báo cáo thực tế trên Drive.
Thư mục báo cáo: `CMU-MOSI/preprocessing/visual/Task_2_BaoCao_Visual`.
File báo cáo giữ tên/số task cũ; số task trong notebook này theo thứ tự mới.

In [ ]:
def show_data_paths(paths=None):
    paths = paths or VisualPaths()
    display(pd.DataFrame([{'file': p.name, 'exists': p.is_file(), 'path': str(p)}
                          for p in [paths.manifest, paths.raw_zip, paths.config, paths.module, paths.backup]]))
    return paths


# Thực hiện
paths = show_data_paths()
def show_report_files(paths):
    root = paths.visual / 'Task_2_BaoCao_Visual'
    print('Thư mục báo cáo:', root)
    if not root.is_dir():
        print('Chưa thấy thư mục báo cáo tại đường dẫn này.')
        return
    display(pd.DataFrame([{'Tên để mở': p.name,
                           'Loại': 'Thư mục' if p.is_dir() else 'File',
                           'Đường dẫn': str(p)} for p in sorted(root.iterdir())]))

show_report_files(paths)


## Task 2.3 — Kiểm tra manifest | Mở File 01 trên Drive
**Nói:** “Em kiểm tra manifest để xác định đúng clip, transcript, nhãn và giữ split của nhóm.”
**Hàm ngay dưới:** `read_manifest()`.
- `pd.read_csv(...)`: đọc `preprocessing/shared/manifest.csv`.
- `manifest.video_id + '$_$' + manifest.clip_id`: kiểm tra ID phát ngôn; `$_$` chỉ là chuỗi phân cách.
- `labels.between(-3,3)`: kiểm tra phạm vi nhãn sentiment; vẫn giữ nhãn 0 và transcript trùng.
- `groupby('video_id').split.nunique()`: phát hiện video nguồn xuất hiện ở nhiều tập.
- `{'train':1284, 'valid':229, 'test':686}`: đối chiếu split, không tự chia lại.
**Đầu ra:** bảng số lỗi và số clip/video theo split. Manifest ghép dữ liệu bằng ID, chưa căn chỉnh theo thời gian.

In [ ]:
def read_manifest(paths, show=True):
    """Kiểm tra đầu vào một lần; giữ nguyên nhãn và split của nhóm."""
    columns = ['id','video_id','clip_id','split','raw_text','label',
               'source_archive','video_relpath','video_size_bytes']
    manifest = pd.read_csv(paths.manifest, dtype={
        'id':str,'video_id':str,'clip_id':str}, keep_default_na=False)
    if not set(columns).issubset(manifest.columns):
        raise ValueError('Manifest thiếu cột cần thiết.')
    labels = pd.to_numeric(manifest.label, errors='coerce')
    sizes = pd.to_numeric(manifest.video_size_bytes, errors='coerce')
    text_columns = ['id','video_id','clip_id','split','raw_text','video_relpath']
    checks = pd.Series({
        'Thông tin thiếu hoặc ID trùng': int(manifest[text_columns].apply(
            lambda s:s.str.strip().eq('')).any(axis=1).sum()+manifest.id.duplicated().sum()),
        'ID không khớp video và clip': int(manifest.id.ne(
            manifest.video_id+'$_$'+manifest.clip_id).sum()),
        'Nhãn không hợp lệ': int((~np.isfinite(labels)|~labels.between(-3,3)).sum()),
        'Nguồn video không hợp lệ': int((~manifest.source_archive.eq('Raw.zip')|
            ~np.isfinite(sizes)|sizes.le(0)).sum()),
        'Video nguồn nằm ở nhiều tập': int(manifest.groupby('video_id').split.nunique().gt(1).sum()),
        'Split khác bản nhóm đã chốt': int(manifest.split.value_counts().to_dict()!=
            {'train':1284,'valid':229,'test':686}),
    }, name='Số lỗi')
    if checks.gt(0).any():
        raise ValueError(f'Đầu vào cần kiểm tra lại:\n{checks}')
    manifest['label'] = labels
    if show:
        display(checks.to_frame())
        display(manifest.groupby('split').agg(samples=('id','size'),videos=('video_id','nunique')))
    return manifest


# Thực hiện
manifest = read_manifest(paths)

## Task 2.4 — Đọc 10 clip pilot
**Hàm:** `read_pilot_manifest()`.
`pd.read_csv(paths.shared/'pilot_manifest.csv')` lấy ID pilot của nhóm; `.loc[pilot_ids]` lấy đúng các clip và thứ tự đó từ manifest.
**Mục đích:** kiểm tra quy trình trên 10 clip trước khi chạy toàn bộ; không chọn lại pilot.

In [ ]:
def read_pilot_manifest(paths, manifest):
    """Lấy đúng các ID pilot do nhóm chọn, không bốc mẫu mới."""
    pilot_ids = pd.read_csv(paths.shared/'pilot_manifest.csv', dtype=str)['id']
    if pilot_ids.empty or pilot_ids.duplicated().any() or not pilot_ids.isin(manifest.id).all():
        raise ValueError('ID pilot rỗng, trùng hoặc không thuộc manifest.')
    return manifest.set_index('id').loc[pilot_ids].reset_index()


# Thực hiện
pilot = read_pilot_manifest(paths, manifest)
display(pilot[['id','split','raw_text']])

## Task 2.5 — Chuẩn bị video của một clip
**Hàm:** `prepare_clip_video()`.
- `row.video_relpath`: tìm đúng video trong `Raw.zip`.
- `shutil.copyfileobj(src, dst)`: lấy video sang bộ nhớ runtime để công cụ đọc.
- `while cap.read()[0]`: đếm toàn bộ frame giải mã được để đối chiếu đầu ra OpenFace.
Chỉ định nghĩa hàm; chưa lấy video cho đến khi gọi. Không lấy riêng vài frame để trích đặc trưng.

In [ ]:
def prepare_clip_video(paths, row):
    require_cv2()
    destination = safe_child(paths.local_root / 'media', f'{row.id}.mp4')
    destination.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(paths.raw_zip) as archive:
        matches = [m for m in archive.infolist() if m.filename == row.video_relpath]
        if len(matches) != 1:
            raise ValueError('Video phải xuất hiện đúng một lần trong ZIP.')
        member = matches[0]
        if member.is_dir() or member.file_size != int(row.video_size_bytes):
            raise ValueError('Kích thước video không khớp manifest.')
        # Ghi lại từ ZIP để không dùng nhầm file local cũ cùng kích thước.
        temporary = destination.with_suffix('.mp4.part')
        with archive.open(member) as src, temporary.open('wb') as dst:
            shutil.copyfileobj(src, dst)
        temporary.replace(destination)
    cap = cv2.VideoCapture(str(destination))
    count = 0
    try:
        if not cap.isOpened():
            raise ValueError('Không mở được video.')
        while cap.read()[0]:
            count += 1
    finally:
        cap.release()
    if not count:
        raise ValueError('Không có frame đọc được.')
    return destination, count


# Thực hiện
print('Đã định nghĩa prepare_clip_video(); chỉ lấy video khi chạy thử.')

## Task 2.6 — Xem môi trường OpenFace
`show_openface_environment()` kiểm tra chương trình, model và ffmpeg có trong runtime hay chưa.
Runtime mới có thể chưa có OpenFace; vẫn có thể đọc kết quả đã lưu.

In [ ]:
def show_openface_environment(paths):
    report = {'platform':platform.platform(), 'binary_exists':paths.binary.is_file(),
              'model_exists':paths.model.is_file(), 'ffmpeg':shutil.which('ffmpeg')}
    display(report)
    return report


# Thực hiện
environment = show_openface_environment(paths)

## Task 2.7 — Xem công cụ hệ thống
`show_system_tools()` dùng `shutil.which(...)` để tìm các chương trình hỗ trợ cài đặt. Chưa cài hoặc trích xuất.

In [ ]:
def show_system_tools():
    if platform.system() != 'Linux':
        raise RuntimeError('Chức năng cài OpenFace dành cho Colab/Linux.')
    report = {name: shutil.which(name) for name in ['apt-get','cmake','git','g++','wget','ffmpeg']}
    display(report)
    return report


# Thực hiện
show_system_tools()

## Task 2.8 — Cài thư viện hệ thống khi cần
`install_system_dependencies()` gọi `apt-get` để cài các thư viện OpenFace phụ thuộc.
Giữ `SETUP_OPENFACE=False` khi chỉ đọc kết quả; cell sẽ in thông báo bỏ qua.

In [ ]:
def install_system_dependencies():
    show_system_tools()
    packages = ['build-essential','cmake','git','wget','pkg-config','libopencv-dev',
                'libdlib-dev','libopenblas-dev','libboost-filesystem-dev','libboost-system-dev','ffmpeg']
    for command in [['apt-get','update','-qq'],
                    ['apt-get','install','-y','--no-install-recommends',*packages]]:
        print('Đang chạy:', ' '.join(command), flush=True)
        subprocess.run(command, check=True, env={**os.environ, 'DEBIAN_FRONTEND':'noninteractive'})


# Thực hiện
if SETUP_OPENFACE:
    install_system_dependencies()
else:
    print('Bỏ qua cài đặt: đang đọc kết quả đã lưu.')

## Task 2.9 — Biên dịch OpenFace khi thiếu bản sao
`build_openface()` lấy đúng `SOURCE_COMMIT`, cấu hình bằng `cmake -S ... -B ...`, rồi biên dịch mục tiêu `FeatureExtraction`.
Chỉ chạy khi bật cài đặt và chưa có binary/backup. Đây là xây dựng công cụ, không phải huấn luyện MAG-BERT.

In [ ]:
def build_openface(paths):
    root = paths.openface_root
    if not root.exists():
        subprocess.run(['git','init',str(root)], check=True)
        subprocess.run(['git','-C',str(root),'remote','add','origin',SOURCE_URL], check=True)
    if not (root / '.git').is_dir():
        raise RuntimeError('OpenFace root có sẵn nhưng không phải source Git; ưu tiên restore.')
    subprocess.run(['git','-C',str(root),'fetch','--depth','1','origin',SOURCE_COMMIT], check=True)
    subprocess.run(['git','-C',str(root),'checkout','--detach',SOURCE_COMMIT], check=True)
    for command in [['cmake','-S',str(root),'-B',str(root/'build'),'-DCMAKE_BUILD_TYPE=Release'],
                    ['cmake','--build',str(root/'build'),'--target','FeatureExtraction','--parallel','2']]:
        subprocess.run(command, check=True)
    if not paths.binary.is_file():
        raise RuntimeError('Build chưa tạo FeatureExtraction.')
    return paths.binary


# Thực hiện
if SETUP_OPENFACE and not paths.binary.is_file() and not paths.backup.is_file():
    build_openface(paths)
else:
    print('Không cần biên dịch OpenFace ở bước này.')

## Task 2.10 — Khôi phục bộ OpenFace khi cần
`restore_or_backup_openface()` dùng bản sao công cụ trên Drive để khôi phục vào runtime.
Giữ `SETUP_OPENFACE=False` nếu chỉ báo cáo. Việc khôi phục không tạo lại đặc trưng của video.

In [ ]:
def restore_or_backup_openface(paths, action='restore'):
    if action == 'restore':
        if paths.binary.is_file() and paths.model.is_file():
            return paths.binary
        if not paths.backup.is_file():
            raise FileNotFoundError(paths.backup)
        if paths.openface_root.name != 'OpenFace':
            raise ValueError('Archive cần openface_root có tên OpenFace.')
        with tarfile.open(paths.backup, 'r:gz') as archive:
            for member in archive.getmembers():
                relative = PurePosixPath(member.name)
                if relative.is_absolute() or '..' in relative.parts or not relative.parts or relative.parts[0] != 'OpenFace':
                    raise ValueError(f'Đường dẫn archive không hợp lệ: {member.name}')
            archive.extractall(paths.openface_root.parent, filter='data')
    elif action == 'backup':
        if not paths.binary.is_file() or not paths.model.is_file():
            raise FileNotFoundError('Chưa có bộ OpenFace đầy đủ để sao lưu.')
        if paths.backup.exists():
            print('Giữ bản sao đã có:', paths.backup)
            return paths.backup
        paths.backup.parent.mkdir(parents=True, exist_ok=True)
        temporary = paths.backup.with_suffix('.gz.part')
        with tarfile.open(temporary, 'w:gz') as archive:
            archive.add(paths.binary.parent, arcname='OpenFace/build/bin')
        temporary.replace(paths.backup)
        return paths.backup
    else:
        raise ValueError('action phải là restore hoặc backup.')
    return paths.binary


# Thực hiện
if SETUP_OPENFACE and not paths.binary.is_file() and paths.backup.is_file():
    restore_or_backup_openface(paths)
else:
    print('Bỏ qua khôi phục công cụ.')

## Task 2.11 — Chốt tên và thứ tự 22 chiều | Mở File 04 trên Drive
**Nói:** “Từ đầu ra OpenFace, em chọn 22 chiều: 17 cường độ AU, 3 góc xoay đầu và 2 góc nhìn.”
**Hàm:** `load_feature_config()` đọc `preprocessing/visual/config/visual_feature_config.json`.
**Ngay trong code dưới:** `AU_NAMES`, `POSE_NAMES`, `GAZE_NAMES`, `FEATURE_NAMES` ghi rõ 22 cột và thứ tự.
- Hậu tố `_r`: cường độ AU do OpenFace ước lượng; AU45 liên quan nháy mắt.
- `pose_Rx`, `pose_Ry`, `pose_Rz`: góc xoay đầu quanh ba trục, đơn vị radian.
- `gaze_angle_x`, `gaze_angle_y`: hai góc hướng nhìn, đơn vị radian.
Đây là cấu hình nhóm chọn từ đầu ra OpenFace, không phải một bộ 22 chiều bắt buộc của OpenFace. MAG-BERT gốc dùng Facet cho visual.
`prepare_openface_models()` chỉ chuẩn bị model nhận diện của OpenFace; không huấn luyện MAG-BERT.

In [ ]:
# 17 cường độ AU, 3 góc xoay đầu, 2 góc nhìn — thứ tự dùng chung mọi clip.
AU_NAMES = [f'AU{i:02d}_r' for i in [1,2,4,5,6,7,9,10,12,14,15,17,20,23,25,26,45]]
POSE_NAMES = ['pose_Rx', 'pose_Ry', 'pose_Rz']
GAZE_NAMES = ['gaze_angle_x', 'gaze_angle_y']
FEATURE_NAMES = AU_NAMES + POSE_NAMES + GAZE_NAMES

def prepare_openface_models(paths, download=False):
    if not paths.binary.is_file() or not paths.model.is_file():
        raise FileNotFoundError('Thiếu FeatureExtraction hoặc main_ceclm_general.txt.')
    folder = paths.binary.parent / 'model/patch_experts'
    for name, url in MODEL_URLS.items():
        target = folder / name
        if not target.is_file() or target.stat().st_size == 0:
            if not download:
                raise FileNotFoundError(target)
            folder.mkdir(parents=True, exist_ok=True)
            temporary = target.with_suffix('.dat.part')
            subprocess.run(['wget','--timeout=60','--tries=2','--no-verbose','-O',str(temporary),url], check=True)
            with temporary.open('rb') as stream:
                header = stream.read(512).lstrip().lower()
            if not temporary.stat().st_size or header.startswith((b'<!doctype html',b'<html',b'<?xml')):
                raise ValueError(f'Model không hợp lệ: {name}')
            temporary.replace(target)
    check = subprocess.run(['ldd',str(paths.binary)], capture_output=True, text=True, check=False)
    if check.returncode or 'not found' in check.stdout:
        raise RuntimeError(f'OpenFace thiếu thư viện: {check.stdout}\n{check.stderr}')
    return paths.model

def load_feature_config(paths, create=False):
    default = {'schema_version':'1.0', 'extractor':'OpenFace', 'feature_dim':22,
               'feature_names':FEATURE_NAMES,
               'feature_groups':{'au_intensity':AU_NAMES,'head_rotation':POSE_NAMES,'gaze_angles':GAZE_NAMES},
               'dtype':'float32', 'timestamp_unit':'seconds',
               'metadata_columns':['frame','timestamp','success','confidence'],
               'selection_basis':'Project configuration using documented OpenFace outputs: 17 AU intensities, 3 head rotation angles, 2 gaze angles.',
               'reference_url':'https://github.com/TadasBaltrusaitis/OpenFace/wiki/Output-Format'}
    if paths.config.is_file():
        config = json.loads(paths.config.read_text(encoding='utf-8'))
    elif create:
        write_json(paths.config, default)
        config = default
    else:
        raise FileNotFoundError(paths.config)
    if config.get('feature_dim') != 22 or config.get('feature_names') != FEATURE_NAMES:
        raise ValueError('Cấu hình không khớp tên/thứ tự 22 chiều đã chốt.')
    return config


# Thực hiện
if SETUP_OPENFACE:
    prepare_openface_models(paths, download=True)
config = load_feature_config(paths)
display(pd.DataFrame({'Thứ tự':range(1,23),'Đặc trưng':config['feature_names']}))

## Task 2.12A — Đọc CSV: chọn 22 cột, float32, thời gian và mask
**Hàm thật dùng trong quy trình:** `read_visual_csv()`; không cần mở file Python khác.
`names = list(feature_names)` nhận thứ tự 22 cột của Task 2.11; `numeric[names]` chọn chúng. Tìm ngay trong cell dưới đoạn `features = ... dtype=np.float32` và `observed_mask = ...`.
`timestamp` lấy thời gian giây, `success` là trạng thái 0/1, `confidence` là độ tin cậy do OpenFace xuất. Thời gian giữ float64; ma trận đặc trưng chuyển float32.
**Không biến giá trị thiếu thành 0, không xóa frame.** Mask phục vụ xử lý thiếu/căn chỉnh sau này, chưa tự giải quyết dữ liệu thiếu.

In [ ]:
def read_visual_csv(csv_path, feature_names):
    """Read OpenFace CSV; preserve frame order and missing values."""
    csv_path = Path(csv_path)

    if not csv_path.is_file():
        raise FileNotFoundError(
            f"Missing OpenFace CSV: {csv_path}"
        )

    names = list(feature_names)

    if not names or any(
        not isinstance(name, str) for name in names
    ):
        raise ValueError(
            "feature_names must be a nonempty list of strings."
        )

    if len(names) != len(set(names)):
        raise ValueError("feature_names contains duplicates.")

    # Đọc bảng OpenFace; mỗi dòng tương ứng một frame.
    table = pd.read_csv(csv_path)
    table.columns = table.columns.str.strip()

    if table.columns.duplicated().any():
        raise ValueError(
            "OpenFace CSV contains duplicate column names."
        )

    if table.empty:
        raise ValueError(
            f"OpenFace CSV has no frames: {csv_path}"
        )

    required = [
        "frame", "timestamp", "success", "confidence"
    ] + names

    missing = [
        name for name in required
        if name not in table.columns
    ]

    if missing:
        raise ValueError(
            f"Missing OpenFace columns: {missing}"
        )

    # Chuyển các cột cần dùng sang dữ liệu số.
    numeric = table[required].apply(
        pd.to_numeric,
        errors="raise",
    )

    frame_values = numeric["frame"].to_numpy(
        dtype=np.float64
    )

    expected_frames = np.arange(
        1, len(table) + 1,
        dtype=np.int64,
    )

    if not np.array_equal(
        frame_values, expected_frames
    ):
        raise ValueError(
            "Frame numbers must be consecutive, from 1 to T."
        )

    timestamps = numeric["timestamp"].to_numpy(
        dtype=np.float64
    )

    if (
        not np.isfinite(timestamps).all()
        or (timestamps < 0).any()
        or (np.diff(timestamps) <= 0).any()
    ):
        raise ValueError(
            "Timestamps must be finite, nonnegative "
            "and increasing."
        )

    success_values = numeric["success"].to_numpy(
        dtype=np.float64
    )

    if not np.isin(success_values, [0, 1]).all():
        raise ValueError(
            "OpenFace success must be 0 or 1."
        )

    success = success_values.astype(np.uint8)

    confidence = numeric["confidence"].to_numpy(
        dtype=np.float32
    )

    if (
        not np.isfinite(confidence).all()
        or (confidence < 0).any()
        or (confidence > 1).any()
    ):
        raise ValueError(
            "OpenFace confidence must be finite "
            "and in [0, 1]."
        )

    # CHỌN 22 CỘT theo thứ tự names và chuyển sang float32.
    # float32 là kiểu số thực 32 bit; KHÔNG phải chuẩn hóa giá trị.
    features = numeric[names].to_numpy(
        dtype=np.float32,
        copy=True,
    )

    # True khi giá trị hữu hạn VÀ OpenFace báo success=1.
    # Không xóa frame, không tự điền thiếu, không đặt ngưỡng confidence.
    observed_mask = (
        np.isfinite(features)
        & (success[:, None] == 1)
    )

    return {
        "features": features,
        "timestamps": timestamps,
        "feature_names": names,
        "success": success,
        "confidence": confidence,
        "observed_mask": observed_mask,
        "frame_numbers": expected_frames,
        "csv_path": str(csv_path),
    }

print("Đã định nghĩa read_visual_csv(): chọn cột, float32, timestamp và mask.")


## Task 2.12B — Gọi OpenFace trên video
**Hàm thật:** `extract_visual_features(video_path, config)`.
Tìm `command = [...]`: `-f` chọn video, `-mloc` chỉ model, `-out_dir` nơi xuất, `-of features` đặt tên CSV; `-aus/-pose/-gaze` yêu cầu ba nhóm đặc trưng.
Tìm `process = subprocess.run(...)`: đây là lúc chương trình OpenFace được chạy.
Tìm `result = read_visual_csv(...)`: đọc `features.csv`, chọn 22 chiều và tạo mask bằng hàm vừa định nghĩa ở Task 2.12A.
**Chạy cell này chỉ định nghĩa hàm, chưa phân tích video.**

In [ ]:
def extract_visual_features(video_path, config):
    """Run OpenFace on one clip and return temporal features.

    Required config:
        openface_bin, model_path, output_dir, feature_names.

    Each call creates a separate folder inside output_dir.
    """
    video_path = Path(
        video_path
    ).expanduser().resolve()

    binary = Path(
        config["openface_bin"]
    ).expanduser().resolve()

    model_path = Path(
        config["model_path"]
    ).expanduser().resolve()

    for path in (video_path, binary, model_path):
        if not path.is_file():
            raise FileNotFoundError(
                f"Missing input/runtime file: {path}"
            )

    names = list(config["feature_names"])

    if (
        not names
        or any(
            not isinstance(name, str)
            for name in names
        )
        or len(names) != len(set(names))
    ):
        raise ValueError(
            "feature_names must contain unique string names."
        )

    if int(
        config.get("feature_dim", len(names))
    ) != len(names):
        raise ValueError(
            "feature_dim does not match feature_names."
        )

    output_root = Path(
        config["output_dir"]
    ).expanduser().resolve()

    output_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    run_dir = Path(
        tempfile.mkdtemp(
            prefix="openface_",
            dir=output_root,
        )
    )

    log_path = run_dir / "openface.log"

    # LỆNH GỌI OPENFACE: -aus (AU), -pose (đầu), -gaze (ánh nhìn).
    # OpenFace xuất nhiều cột; read_visual_csv mới chọn 22 cột.
    command = [
        str(binary),
        "-f", str(video_path),
        "-mloc", str(model_path),
        "-out_dir", str(run_dir),
        "-of", "features",
        "-aus", "-pose", "-gaze",
    ]

    (run_dir / "command.json").write_text(
        json.dumps(
            {
                "video_path": str(video_path),
                "command": command,
            },
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

    try:
        with log_path.open(
            "w", encoding="utf-8"
        ) as log:
            # ĐÂY là chỗ thực thi chương trình OpenFace trên video.
            process = subprocess.run(
                command,
                cwd=str(binary.parent),
                stdout=log,
                stderr=subprocess.STDOUT,
                timeout=config.get(
                    "timeout_seconds"
                ),
                check=False,
            )

    except (
        OSError,
        subprocess.TimeoutExpired,
    ) as error:
        raise RuntimeError(
            "OpenFace could not finish; "
            f"inspect {log_path}"
        ) from error

    if process.returncode != 0:
        raise RuntimeError(
            f"OpenFace exit code {process.returncode}; "
            f"inspect {log_path}"
        )

    # Sau khi OpenFace chạy xong, đọc CSV và chọn 22 chiều.
    result = read_visual_csv(
        run_dir / "features.csv",
        names,
    )

    result.update({
        "video_path": str(video_path),
        "log_path": str(log_path),
        "run_dir": str(run_dir),
    })

    return result

print("Đã định nghĩa extract_visual_features(); chưa gọi OpenFace.")


## Task 2.12C — Cung cấp hai hàm cho các task điều phối
`get_visual_processor()` trả về hai hàm đã nhìn thấy ở trên. `processor.extract_visual_features(...)` ở Task 2.14 và 2.22 sẽ gọi đúng hàm tại Task 2.12B.
Hai hàm này là code của dự án để gọi/đọc OpenFace, không phải hàm có sẵn của thư viện OpenFace. Giữ thuật toán cùng bản module gốc đã trích dữ liệu.

In [ ]:
def get_visual_processor(paths=None):
    """Dùng đúng hai hàm nhìn thấy ở Task 2.12A–B trong notebook này."""
    return types.SimpleNamespace(
        read_visual_csv=read_visual_csv,
        extract_visual_features=extract_visual_features,
    )

processor = get_visual_processor(paths)
print('Đã sẵn sàng hai hàm trong notebook; chưa chạy OpenFace.')


## Task 2.13 — Thống kê chất lượng frame
**Hàm:** `summarize_frame_quality()`.
- `(result['success'] == 0).sum()`: đếm frame theo dõi thất bại.
- `(~mask).sum()`: đếm số giá trị đặc trưng không được đánh dấu hợp lệ, không phải số clip hoặc số frame.
- Có frame thất bại hoặc giá trị không hợp lệ → `needs_review`; còn lại → `ok`.
Không dùng ngưỡng confidence trong tiêu chí này; không tự xóa clip. `ok` chưa bảo đảm đang theo dõi đúng người nói.

In [ ]:
def summarize_frame_quality(result):
    x = result['features']
    mask = result['observed_mask']
    failed = int((result['success'] == 0).sum())
    missing = int((~mask).sum())
    return {'T_vision':len(x), 'D_vision':x.shape[1], 'failed_frames':failed,
            'unobserved_values':missing, 'min_confidence':float(result['confidence'].min()),
            'quality_flag':'needs_review' if failed or missing else 'ok'}


# Thực hiện
print('summarize_frame_quality(): thống kê từ features, success và observed_mask.')

## Task 2.14 — Ghép luồng xử lý một clip
**Hàm:** `extract_one_clip()`.
1. `prepare_clip_video(paths, row)`: lấy video, đếm frame gốc.
2. `get_visual_processor(paths)`: lấy hai hàm đã định nghĩa rõ ở Task 2.12A–B.
3. `processor.extract_visual_features(video_path, config)`: chạy OpenFace → đọc CSV → chọn 22 cột → tạo mask.
4. `len(result['features']) != decoded`: phát hiện số frame đầu ra không khớp video.
5. `summarize_frame_quality(result)`: thống kê chất lượng.
**Định nghĩa chi tiết có `-aus`, `subprocess.run`, `float32` nằm ở Task 2.12A–B; cell này điều phối gọi chúng.**

In [ ]:
def extract_one_clip(paths, row, output_dir=None):
    video_path, decoded = prepare_clip_video(paths, row)
    processor = get_visual_processor(paths)
    config = dict(load_feature_config(paths))
    config.update(openface_bin=str(paths.binary.resolve()), model_path=str(paths.model.resolve()),
                  output_dir=str(output_dir or (paths.visual / 'pilot' / row.id)))
    result = processor.extract_visual_features(video_path, config)
    if len(result['features']) != decoded:
        raise ValueError('Số frame CSV không khớp số frame giải mã.')
    display(summarize_frame_quality(result))
    return result


# Thực hiện
print('extract_one_clip(): dùng trong vòng lặp 10 clip ở Task 2.15.')
# Muốn kiểm tra riêng một clip: extract_one_clip(paths, pilot.iloc[0])


## Task 2.15 — Chạy hoặc đọc 10 clip | Mở File 02 trên Drive
**Nói:** “Em chạy thử 10 clip, kiểm tra số frame và trạng thái theo dõi trước khi chạy toàn bộ.”
**Hàm:** `extract_pilot_clips()`; `for row in pilot.itertuples(...)` xử lý lần lượt các clip, gọi `extract_one_clip()` ở Task 2.14.
`pilot_quality_summary.csv` lưu thống kê; `progress.jsonl` ghi tiến độ.
**Hiện tại:** `RUN_PILOT=False` → đọc báo cáo pilot cũ và in bảng; không chạy lại OpenFace.
Kết quả gốc: `preprocessing/visual/pilot_batches/20261102T070035_074030Z/`.

In [ ]:
def extract_pilot_clips(paths, manifest):
    pilot = read_pilot_manifest(paths, manifest)
    run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    root = paths.visual / 'pilot_batches' / run_id
    root.mkdir(parents=True, exist_ok=False)
    records = []
    try:
        for row in pilot.itertuples(index=False):
            try:
                result = extract_one_clip(paths, row, root / 'openface' / row.id)
                record = {'id':row.id, 'status':'extracted', 'csv_file':str(result['csv_path']),
                          **summarize_frame_quality(result)}
            except Exception as error:
                record = {'id':row.id, 'status':'error', 'error':f'{type(error).__name__}: {error}'}
            records.append(record)
            with (root / 'progress.jsonl').open('a', encoding='utf-8') as stream:
                stream.write(json.dumps(record, ensure_ascii=False) + '\n')
    finally:
        pd.DataFrame(records).to_csv(root / 'pilot_quality_summary.csv', index=False)
    return pd.DataFrame(records)


# Thực hiện
if RUN_PILOT:
    pilot_report = extract_pilot_clips(paths, manifest)
else:
    pilot_report = pd.read_csv(paths.visual/'pilot_batches'/PILOT_RUN/'pilot_quality_summary.csv')
display(pilot_report[['id','status']])

## Task 2.16 — Xem frame minh họa | Mở thư mục 03 trên Drive
**Hàm:** `show_sample_frames()`; `[1,(count+1)//2,count]` lấy frame đầu, giữa và cuối để xem.
**Mục đích:** kiểm tra bằng mắt một vài thời điểm; OpenFace vẫn trích đặc trưng toàn bộ frame.
Hàm này chỉ hiển thị ảnh gốc. Ảnh có lớp minh họa OpenFace đã lưu nằm trong `03_Task_2_16_17_Anh_OpenFace`; không nói hàm này tự vẽ landmark.
Mặc định chỉ in đường dẫn ảnh đã lưu; không đọc video lại.

In [ ]:
def show_sample_frames(video_path, frame_numbers=None):
    require_cv2()
    cap = cv2.VideoCapture(str(video_path))
    count = 0
    try:
        if not cap.isOpened():
            raise ValueError('Không mở được video.')
        while cap.read()[0]:
            count += 1
    finally:
        cap.release()
    if not count:
        raise ValueError('Video không có frame.')
    selected = frame_numbers or sorted(set([1,(count+1)//2,count]))
    if any(not isinstance(i, int) or i < 1 or i > count for i in selected):
        raise ValueError('Chỉ số frame phải là số nguyên trong [1, T].')
    selected = sorted(set(selected))
    result = []
    cap = cv2.VideoCapture(str(video_path))
    try:
        for index in range(1, max(selected)+1):
            ok, frame = cap.read()
            if not ok:
                raise ValueError('Video lỗi khi đọc lại các frame minh họa.')
            if index in selected:
                result.append({'frame':index, 'rgb':cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)})
    finally:
        cap.release()
    if plt is not None:
        fig, axes = plt.subplots(1, len(result), figsize=(5*len(result), 4), squeeze=False)
        for axis, item in zip(axes[0], result):
            axis.imshow(item['rgb'])
            axis.set_title(f"Frame {item['frame']}", parse_math=False)
            axis.axis('off')
        fig.tight_layout()
        plt.show()
        plt.close(fig)
    return result


# Thực hiện
print('Ảnh đã lưu: preprocessing/visual/Task_2_BaoCao_Visual/03_Task_2_16_17_Anh_OpenFace')
# Khi cần: video_path, _ = prepare_clip_video(paths, pilot.iloc[0])
# show_sample_frames(video_path)

## Task 2.17 — Ghi nhận quan sát trực quan | Mở thư mục 03 trên Drive
**Hàm:** `create_visual_review_form()` tạo bảng ghi chú.
Các cột chính: `speaker_visible`, `occlusion_or_multiple_faces`, `tracking_notes`.
**Người xem** ghi nhận che mặt, chuyển cảnh hoặc theo dõi lệch; hàm không tự nhận xét. `not_reviewed` nghĩa là chưa có đánh giá trong bảng mới, không phải kết luận clip lỗi.

In [ ]:
def create_visual_review_form(ids, output_path=None):
    review = pd.DataFrame({'id':list(ids), 'speaker_visible':'', 'occlusion_or_multiple_faces':'',
                           'tracking_notes':'', 'review_decision':'not_reviewed'})
    if output_path is not None:
        path = Path(output_path)
        if path.exists():
            raise FileExistsError(f'Giữ ghi chú đã có: {path}')
        path.parent.mkdir(parents=True, exist_ok=True)
        review.to_csv(path, index=False)
    return review


# Thực hiện
review_form = create_visual_review_form(pilot.id)
display(review_form.head(2))

## Task 2.18 — Xem 22 chiều, thời gian, float32 và mask | Mở File 04
**Nói:** “Em giữ đặc trưng theo từng frame cùng mốc thời gian và mask hợp lệ để phục vụ căn chỉnh với từng từ.”
**Hàm:** `save_pilot_features()` gọi `read_visual_csv()` ở Task 2.12A.
**Minh chứng code thực tế tại Task 2.12A:**
- `numeric[names]`: lấy đúng các cột theo tên/thứ tự đã chốt.
- `.to_numpy(dtype=np.float32, copy=True)`: chuyển sang ma trận số thực 32 bit; **không phải chuẩn hóa**. float32 dùng 4 byte/giá trị, float64 dùng 8 byte/giá trị.
- `numeric['timestamp']`: lấy thời gian từng frame, đơn vị giây; kiểm tra tăng dần bằng `np.diff(timestamps)`.
- `np.isfinite(features) & (success[:, None] == 1)`: tạo mask T × 22. True khi giá trị hữu hạn và frame được báo theo dõi thành công.
`show_visual_values()` ngay dưới sẽ đọc một CSV pilot đã lưu để hiện **giá trị thật**, shape, dtype, thời gian và mask. Không gọi OpenFace, không ghi đè file.
Chưa điền thiếu, chuẩn hóa hoặc căn chỉnh với từng từ.

In [ ]:
def save_pilot_features(paths, csv_path, output_path=None):
    config = load_feature_config(paths)
    result = get_visual_processor(paths).read_visual_csv(csv_path, config['feature_names'])
    if output_path is not None:
        output = Path(output_path)
        if output.exists():
            raise FileExistsError(f'Không ghi đè pilot: {output}')
        output.parent.mkdir(parents=True, exist_ok=True)
        with output.with_suffix(output.suffix + '.part').open('wb') as stream:
            np.savez_compressed(stream, **{k:np.asarray(result[k]) for k in
                ['features','timestamps','feature_names','success','confidence','observed_mask','frame_numbers']})
        output.with_suffix(output.suffix + '.part').replace(output)
    return result


# Thực hiện
print('save_pilot_features(paths, csv_path, output_path): đọc CSV và lưu đặc trưng khi cần.')
def show_visual_values(result, title='Dữ liệu visual thực tế'):
    features = np.asarray(result['features'])
    mask = np.asarray(result['observed_mask'])
    names = list(result['feature_names'])
    print(title)
    print('Shape:', features.shape, '| dtype:', features.dtype,
          '| byte/giá trị:', features.dtype.itemsize)
    print('Số frame:', len(features), '| số chiều:', features.shape[1])
    print('Ví dụ 3 frame đầu, 22 cột theo đúng thứ tự:')
    display(pd.DataFrame(features[:3], columns=names))
    print('Thời gian và trạng thái của 3 frame đó:')
    display(pd.DataFrame({
        'frame': np.asarray(result['frame_numbers'])[:3],
        'timestamp_seconds': np.asarray(result['timestamps'])[:3],
        'success': np.asarray(result['success'])[:3],
        'confidence': np.asarray(result['confidence'])[:3],
        'valid_feature_values': mask[:3].sum(axis=1),
    }))
    print('Mask tương ứng (True = được đánh dấu hợp lệ):')
    display(pd.DataFrame(mask[:3], columns=names))

# Chỉ đọc CSV pilot cũ; không trích xuất lại, không ghi file.
existing_pilot_csv = next((Path(p) for p in pilot_report.get(
    'csv_file', pd.Series(dtype=str)).dropna().astype(str)
    if p and Path(p).is_file()), None)
if existing_pilot_csv is not None:
    pilot_example = save_pilot_features(paths, existing_pilot_csv)
    print('CSV nguồn:', existing_pilot_csv)
    show_visual_values(pilot_example, 'Minh chứng từ CSV pilot đã lưu')
else:
    print('Chưa tìm thấy CSV pilot theo đường dẫn báo cáo; '
          'Task 2.21 sẽ đọc một NPZ kết quả đầy đủ để minh họa.')


## Task 2.19 — Lệnh Git tham khảo
`show_git_commands()` chỉ in lệnh bàn giao mã. Chạy cell không tự commit/push.
Notebook/mã lên GitHub; dữ liệu đặc trưng đã trích vẫn cần Drive để nhóm dùng lại.

In [ ]:
def show_git_commands():
    print('Trong checkout DataMining-MSA, đặt notebook này tại src/notebooks/.')
    print('git switch feat/preprocess-visual')
    print('git add src/notebooks/02_Visual_BaoHoang_BaoCao.ipynb')
    print('git diff --cached --stat')
    print('git commit -m "Consolidate visual preprocessing tasks 2.1-2.22"')
    print('git push origin feat/preprocess-visual')
    print('Hậu pull nhánh này; Drive cần được chia sẻ riêng để đọc NPZ và báo cáo.')


# Thực hiện
show_git_commands()

## Task 2.20 — Đối chiếu công cụ trước khi trích tiếp
`check_extraction_runtime()` so sánh cấu hình 22 cột, phiên bản module gốc và các file công cụ với lần trích trước bằng `SAVED_SIGNATURE`.
Hai hàm hiển thị trong notebook là bản cùng thuật toán với module gốc đã trích dữ liệu; đã đối chiếu khi tạo notebook này.
Chỉ đọc báo cáo thì bỏ qua. Không sửa thuật toán rồi dùng chung cache cũ.

In [ ]:
def check_extraction_runtime(paths):
    """Đối chiếu một cấu hình trước khi tiếp tục cache cũ."""
    load_feature_config(paths)
    prepare_openface_models(paths)
    runtime = {str(p.relative_to(paths.binary.parent)):file_sha256(p)
               for p in sorted(paths.binary.parent.rglob('*')) if p.is_file()}
    recipe = {'schema_version':'visual22-full-v1','feature_names':FEATURE_NAMES,
              'processor_sha256':file_sha256(paths.module),'openface_runtime_sha256':runtime}
    signature = hashlib.sha256(json.dumps(recipe,sort_keys=True).encode('utf-8')).hexdigest()
    if signature != SAVED_SIGNATURE:
        raise ValueError('Bộ trích xuất khác cấu hình cũ; dừng để không trộn kết quả.')
    print('Bộ trích xuất khớp kết quả đã lưu:',signature[:12])
    return recipe


# Thực hiện
if RUN_EXTRACTION:
    recipe = check_extraction_runtime(paths)
else:
    print('Đọc kết quả cũ không cần kiểm tra runtime hoặc tải OpenFace.')

## Task 2.21 — Đọc kết quả đầy đủ | Mở File 06 trên Drive
**Nói:** “Em đã trích đủ 2.199 clip; kiểm tra tự động ghi nhận 2.139 đạt và 60 cần xem thêm. Em giữ toàn bộ mẫu.”
**Hàm:** `read_saved_results()` đọc `visual_manifest.csv` đã lưu, đối chiếu ID/nhãn/split và kiểm tra file đặc trưng có tồn tại.
`report.groupby(['split','quality_flag']).size()` thống kê số clip đạt/cần xem thêm theo split.
`validate_feature_file()` chỉ dùng để kiểm tra sâu từng file khi `VERIFY_SAVED_FEATURES=True`; mặc định False.
Cell cũng đọc **một file đặc trưng thật** để hiển thị shape, float32, timestamp và mask; không chạy OpenFace.
Báo cáo gốc: `preprocessing/visual/full_visual_22/3c908e7632a1/runs/20261102T140801_205739Z/visual_manifest.csv`.
**Ví dụ xem video trong thư mục 03:** `8qrpnFRGt2A$_$13` người nói cúi xuống không thấy mặt; `5W7Z1C_fDaE$_$24` chuyển sang nội dung quảng bá. Hai nhận xét này từ quan sát thủ công, không phải hàm tự suy ra.

In [ ]:
def validate_feature_file(path, row, member):
    """Kiểm tra cache trước khi dùng lại: đúng mẫu, cấu hình, shape, thời gian và mask."""
    with np.load(path, allow_pickle=False) as saved:
        expected = {'id':row.id,'signature':SAVED_SIGNATURE,
                    'source_member':row.video_relpath,'source_crc32':member.CRC,
                    'source_size':member.file_size}
        if any(saved[k].item()!=v for k,v in expected.items()):
            raise ValueError('File không khớp clip, nguồn video hoặc cấu hình.')
        x, ts = saved['features'], saved['timestamps']
        success, mask = saved['success'], saved['observed_mask']
        count = len(x)
        if (x.ndim!=2 or count==0 or x.shape[1]!=22 or x.dtype!=np.float32
            or saved['feature_names'].tolist()!=FEATURE_NAMES):
            raise ValueError('Ma trận/tên 22 đặc trưng không đúng.')
        if (ts.shape!=(count,) or not np.isfinite(ts).all() or (ts<0).any()
            or (np.diff(ts)<=0).any() or not np.array_equal(saved['frame_numbers'],np.arange(1,count+1))
            or int(saved['decoded_frames'])!=count):
            raise ValueError('Frame hoặc mốc thời gian không đúng.')
        if (success.shape!=(count,) or not np.isin(success,[0,1]).all()
            or mask.dtype!=np.bool_ or not np.array_equal(mask,np.isfinite(x)&(success[:,None]==1))):
            raise ValueError('Trạng thái theo dõi hoặc mask không đúng.')
        confidence = saved['confidence']
        if confidence.shape!=(count,) or not np.isfinite(confidence).all() or not ((0<=confidence)&(confidence<=1)).all():
            raise ValueError('Confidence không đúng.')
        failed, missing = int((success==0).sum()), int((~mask).sum())
        return {'T_vision':count,'D_vision':22,'failed_frames':failed,
                'unobserved_values':missing,'min_confidence':float(confidence.min()),
                'quality_flag':'needs_review' if failed or missing else 'ok',
                'csv_file':saved['csv_file'].item(),'log_file':saved['log_file'].item()}

def read_saved_results(paths, verify=False, run_id=COMPLETED_RUN):
    """Đọc kết quả hiện có; không cần cài hoặc chạy OpenFace."""
    manifest = read_manifest(paths, show=False)
    root = paths.full_root
    config = json.loads((root/'extraction_config.json').read_text(encoding='utf-8'))
    recipe = config['recipe']
    signature = hashlib.sha256(json.dumps(recipe,sort_keys=True).encode('utf-8')).hexdigest()
    if signature!=SAVED_SIGNATURE or config['signature']!=SAVED_SIGNATURE:
        raise ValueError('Báo cáo thuộc cấu hình trích xuất khác.')
    report_path = safe_child(root/'runs',run_id)/'visual_manifest.csv'
    report = pd.read_csv(report_path,dtype={'id':str},keep_default_na=False)
    if report.id.duplicated().any() or set(report.id)!=set(manifest.id):
        raise ValueError('Danh sách clip trong kết quả không khớp manifest.')
    joined = manifest[['id','split','label']].merge(report[['id','split','label']],on='id',suffixes=('_source','_saved'))
    if not joined.split_source.eq(joined.split_saved).all() or not np.array_equal(joined.label_source,joined.label_saved):
        raise ValueError('Split hoặc nhãn trong kết quả khác manifest.')
    if not report.status.isin(['cached','extracted']).all():
        raise ValueError('Báo cáo vẫn có clip lỗi hoặc chưa xử lý.')
    feature_paths = {row.id:safe_child(root,row.feature_file) for row in report.itertuples(index=False)}
    if any(path!=safe_child(root/'features',identifier+'.npz') or not path.is_file()
           for identifier,path in feature_paths.items()):
        raise FileNotFoundError('Thiếu file đặc trưng hoặc đường dẫn không đúng.')
    if verify:
        by_id = manifest.set_index('id')
        reported = report.set_index('id')
        for identifier,path in tqdm(feature_paths.items(),desc='Kiểm tra file đã lưu'):
            row = types.SimpleNamespace(id=identifier,**by_id.loc[identifier].to_dict())
            with np.load(path,allow_pickle=False) as saved:
                member = types.SimpleNamespace(CRC=saved['source_crc32'].item(),file_size=int(row.video_size_bytes))
            stats = validate_feature_file(path,row,member)
            if any(stats[k]!=reported.loc[identifier,k] for k in ['T_vision','D_vision','failed_frames','unobserved_values','quality_flag']):
                raise ValueError('Thống kê file khác báo cáo: '+identifier)
    display(report.groupby(['split','quality_flag']).size().unstack(fill_value=0))
    print('Đọc',len(report),'clip từ:',report_path)
    return {'manifest':manifest,'report':report,'feature_paths':feature_paths,
            'feature_names':FEATURE_NAMES.copy(),'full_root':root,'report_path':report_path}


# Thực hiện
bundle = read_saved_results(paths, verify=VERIFY_SAVED_FEATURES)
report = bundle['report']
# Bảng tổng hợp và một ví dụ THẬT từ file đã trích, không chạy lại OpenFace.
print('Tổng số clip:', len(report))
display(report['quality_flag'].value_counts().rename_axis('Tiêu chí').to_frame('Số clip'))
example_id = next(iter(bundle['feature_paths']))
example_path = bundle['feature_paths'][example_id]
with np.load(example_path, allow_pickle=False) as saved:
    example_values = {key: saved[key].copy() for key in [
        'features', 'feature_names', 'frame_numbers', 'timestamps',
        'success', 'confidence', 'observed_mask']}
print('ID minh họa:', example_id)
print('File nguồn:', example_path)
show_visual_values(example_values, 'Minh chứng từ một file kết quả đã lưu')


## Task 2.22 — Tiếp tục trích xuất khi cần
**Hàm:** `process_full_dataset()` chọn đọc kết quả hay trích xuất; `extract_full_dataset()` điều phối toàn bộ clip.
- `cached_stats = validate_feature_file(...)`: kiểm tra file đã có trước khi dùng lại.
- `record['status'] = 'cached'`: file hợp lệ được dùng lại, không trích lại.
- `processor_full.extract_visual_features(...)`: chỉ gọi OpenFace cho file thiếu/hỏng.
- `progress.jsonl`: ghi tiến độ từng clip; `finally` lưu bảng tổng hợp khi kết thúc hoặc gặp lỗi.
**Mặc định RUN_EXTRACTION=False:** chỉ thông báo kết quả đã đọc ở Task 2.21, không xử lý lại 2.199 clip.
**Hướng tiếp theo:** xử lý dữ liệu thiếu, chuẩn hóa bằng thông số từ train, lấy thời gian từng từ bằng MFA (audio–transcript), tổng hợp audio/visual theo khoảng từ rồi ánh xạ token BERT. Chưa thực hiện các bước này và chưa huấn luyện MAG-BERT.

In [ ]:
def extract_full_dataset(paths):
    """Lưu theo từng clip, dùng lại file hợp lệ khi Colab bị ngắt."""
    full_manifest = read_manifest(paths, show=False)
    feature_config = load_feature_config(paths)
    feature_names_full = feature_config['feature_names']
    processor_full = get_visual_processor(paths)
    recipe = check_extraction_runtime(paths)
    recipe_signature = SAVED_SIGNATURE
    RAW_ZIP, OPENFACE_BIN, MODEL_PATH = paths.raw_zip, paths.binary, paths.model
    FULL_ROOT = paths.full_root
    FEATURE_DIR = FULL_ROOT/'features'
    LOCAL_VIDEO_DIR = paths.local_root/'full/videos'
    run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    RUN_DIR = FULL_ROOT/'runs'/run_id
    for folder in [FEATURE_DIR,LOCAL_VIDEO_DIR,RUN_DIR]:
        folder.mkdir(parents=True,exist_ok=True)
    provenance_path = FULL_ROOT/'extraction_config.json'
    if provenance_path.exists():
        previous = json.loads(provenance_path.read_text(encoding='utf-8'))
        if previous['signature']!=recipe_signature or previous['recipe']!=recipe:
            raise ValueError('Cấu hình đã lưu không khớp; không trộn dữ liệu.')
    else:
        write_json(provenance_path,{'signature':recipe_signature,'recipe':recipe,'feature_config':feature_config})
    full_manifest.to_csv(RUN_DIR/'source_manifest.csv',index=False,encoding='utf-8')
    print('Tiếp tục xử lý; file hợp lệ được dùng lại:',FULL_ROOT)

    def decoded_frame_count(video_path):
        cap = cv2.VideoCapture(str(video_path))
        try:
            if not cap.isOpened():
                raise ValueError("Không mở được video.")
            count = 0
            while True:
                ok, _ = cap.read()
                if not ok:
                    break
                count += 1
            if count == 0:
                raise ValueError("Video không có frame đọc được.")
            return count
        finally:
            cap.release()

    def feature_payload(result, row, member, decoded_frames):
        if result["features"].shape[0] != decoded_frames:
            raise ValueError("Số frame OpenFace khác số frame đọc từ video.")
        return {
            "id": np.asarray(row.id),
            "features": result["features"],
            "timestamps": result["timestamps"],
            "feature_names": np.asarray(result["feature_names"], dtype=np.str_),
            "success": result["success"],
            "confidence": result["confidence"],
            "observed_mask": result["observed_mask"],
            "frame_numbers": result["frame_numbers"],
            "decoded_frames": np.asarray(decoded_frames, dtype=np.int64),
            "signature": np.asarray(recipe_signature),
            "source_member": np.asarray(row.video_relpath),
            "source_crc32": np.asarray(member.CRC, dtype=np.uint32),
            "source_size": np.asarray(member.file_size, dtype=np.int64),
            "csv_file": np.asarray(str(
                Path(result["csv_path"]).resolve().relative_to(FULL_ROOT.resolve())
            )),
            "log_file": np.asarray(str(
                Path(result["log_path"]).resolve().relative_to(FULL_ROOT.resolve())
            )),
        }

    records_full = []

    JOURNAL_PATH = RUN_DIR / "progress.jsonl"

    REPORT_PATH = RUN_DIR / "visual_manifest.csv"

    try:
        with zipfile.ZipFile(RAW_ZIP, "r") as archive:
            progress = tqdm(
                full_manifest.itertuples(index=False),
                total=len(full_manifest), desc="Visual MOSI",
            )
            for row in progress:
                started = time.perf_counter()
                record = {
                    "id": row.id, "status": "error",
                    "quality_flag": "not_checked", "feature_file": "",
                    "cache_note": "", "error": "",
                }
                try:
                    member = archive.getinfo(row.video_relpath)
                    if member.is_dir() or member.file_size != int(row.video_size_bytes):
                        raise ValueError("Thông tin video trong ZIP không khớp.")
                    destination = FEATURE_DIR / f"{row.id}.npz"
                    cached_stats = None
                    if destination.is_file():
                        try:
                            cached_stats = validate_feature_file(destination, row, member)
                        except Exception as error:
                            record["cache_note"] = (
                                f"Tạo lại file: {type(error).__name__}: {error}"
                            )
                    if cached_stats is not None:
                        record.update(cached_stats)
                        record["status"] = "cached"
                    else:
                        local_video = LOCAL_VIDEO_DIR / f"{row.id}.mp4"
                        with archive.open(member) as source:
                            with local_video.open("wb") as target:
                                shutil.copyfileobj(source, target)
                        decoded_frames = decoded_frame_count(local_video)
                        run_config = dict(feature_config)
                        run_config.update({
                            "openface_bin": str(OPENFACE_BIN),
                            "model_path": str(MODEL_PATH),
                            "output_dir": str(RUN_DIR / "openface" / row.id),
                        })
                        result = processor_full.extract_visual_features(
                            local_video, run_config
                        )
                        payload = feature_payload(
                            result, row, member, decoded_frames
                        )
                        temporary = destination.with_name(destination.name + f".{run_id}.part")
                        with temporary.open("wb") as stream:
                            np.savez_compressed(stream, **payload)
                        validate_feature_file(temporary, row, member)
                        temporary.replace(destination)
                        record.update(validate_feature_file(destination, row, member))
                        record["status"] = "extracted"

                    record["feature_file"] = str(destination.relative_to(FULL_ROOT))

                except Exception as error:
                    record["error"] = f"{type(error).__name__}: {error}"
                    (RUN_DIR / f"{row.id}.error.txt").write_text(
                        traceback.format_exc(), encoding="utf-8"
                    )

                record["elapsed_seconds"] = round(time.perf_counter() - started, 3)
                records_full.append(record)
                with JOURNAL_PATH.open("a", encoding="utf-8") as journal:
                    journal.write(json.dumps(record, ensure_ascii=False) + "\n")
                progress.set_postfix(
                    last_status=record["status"], quality=record["quality_flag"]
                )

    finally:
        processed = (
            pd.DataFrame(records_full) if records_full
            else pd.DataFrame(columns=["id", "status", "quality_flag"])
        )
        full_report = full_manifest.merge(
            processed, on="id", how="left", validate="one_to_one"
        )
        full_report["status"] = full_report["status"].fillna("pending")
        full_report["quality_flag"] = full_report["quality_flag"].fillna("not_checked")
        temporary_report = REPORT_PATH.with_suffix(".csv.part")
        full_report.to_csv(temporary_report, index=False, encoding="utf-8")
        temporary_report.replace(REPORT_PATH)
        display(
            full_report.groupby(["split", "status"]).size().unstack(fill_value=0)
        )
        print("Tổng hợp trạng thái:")
        print(full_report["status"].value_counts().to_string())
        print("Chất lượng kỹ thuật:")
        print(full_report["quality_flag"].value_counts().to_string())
        print("Manifest kết quả:", REPORT_PATH)
        print("Nhật ký lần chạy:", JOURNAL_PATH)
    return {'report':full_report,'report_path':REPORT_PATH,'full_root':FULL_ROOT}

def process_full_dataset(paths, extract=False, verify=False):
    """Mặc định đọc kết quả. extract=True mới tiếp tục trích xuất."""
    if extract:
        return extract_full_dataset(paths)
    return read_saved_results(paths,verify=verify)


# Thực hiện
if RUN_EXTRACTION:
    bundle = process_full_dataset(paths, extract=True)
    report = bundle['report']
else:
    print('Đã đọc',len(report),'clip ở Task 2.21. Không chạy lại OpenFace.')

## Chỉ nhanh khi thầy hỏi code ở đâu

| Câu hỏi | Mở task | Dòng tìm trong code |
|---|---|---|
| Ghép clip với nhãn thế nào? | 2.3 | `manifest.video_id+'$_$'+manifest.clip_id` |
| 22 chiều cụ thể ở đâu? | 2.11 | `AU_NAMES`, `POSE_NAMES`, `GAZE_NAMES` |
| Chỗ chọn 22 cột? | 2.12A | `features = numeric[names].to_numpy(...)` |
| float32 ở đâu, để làm gì? | 2.12A | `dtype=np.float32`: kiểu số thực 32 bit, không phải chuẩn hóa |
| Mốc thời gian và kiểm tra thứ tự? | 2.12A | `timestamps` và `np.diff(timestamps)` |
| Mask tạo thế nào? | 2.12A | `np.isfinite(features) & (success[:, None] == 1)` |
| Gọi công cụ ở đâu? | 2.12B | `command` và `subprocess.run(...)` |
| Chạy 10 clip ở đâu? | 2.15 | `for row in pilot.itertuples(...)` |
| Vì sao lấy vài frame minh họa? | 2.16 | `[1,(count+1)//2,count]`; chỉ xem ảnh |
| Clip cần xem thêm dựa vào đâu? | 2.13 | `failed`, `missing`, `quality_flag` |
| Đọc số liệu thật đã lưu? | 2.21 | `read_saved_results()` và `groupby` |
| Chạy tiếp không làm lại toàn bộ? | 2.22 | `validate_feature_file`, trạng thái `cached` |

**Đã có:** trích và kiểm tra visual theo frame. **Chưa có:** chuẩn hóa, điền thiếu, căn chỉnh theo từ/token và huấn luyện MAG-BERT.
